<a href="https://colab.research.google.com/github/hsuancheyang/115-1-AI_Fundamentals/blob/main/%E5%B0%8D%E8%A9%B1%E6%A9%9F%E5%99%A8%E4%BA%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 設定 OpenRouter API 金鑰

你需要一個 API 金鑰才能使用 OpenRouter 服務。如果您還沒有，請在 [OpenRouter 網站](https://openrouter.ai/keys) 中建立一個金鑰。

在 Colab 中，您可以點擊左側面板中的 "🔑" 圖標，將金鑰儲存到秘密管理器中。請將其命名為 `OpenRouter`。然後，您可以像下面程式那樣將金鑰傳遞給 SDK：

In [ ]:
# 導入 Python SDK (現在我們將使用 OpenAI 兼容的 API)
from google.colab import userdata

OPENROUTER_API_KEY=userdata.get('OpenRouter')

In [ ]:
!pip install -q openai

## 2. 初始化 google/gemma-4-26b-a4b-it:free 模型

在進行任何 API 呼叫之前，您需要初始化生成模型。我們將使用 `google/gemma-4-26b-a4b-it:free` 模型。

In [ ]:
from openai import OpenAI

# Initialize the OpenAI client with OpenRouter's API base and the API key
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY, # This variable is defined in cell d893953e
)

# Define the model name specific to OpenRouter
model_name = 'google/gemma-4-26b-a4b-it:free'

## 3. 建立網頁版對話機器人介面 (使用 Gradio)

使用 `gradio` 函式庫來建立一個簡單的網頁介面，可以輸入系統提示`system_prompt`（人設）、調整 `top_p` (p 值) 和 `temperature` (回應溫度)，並進行多輪對話。

首先，安裝 `gradio`：

In [ ]:
!pip install -q gradio

接下來是聊天機器人的程式碼。它包含一個 `chat_session` 函數，用於處理對話邏輯，以及一個 `gradio` 介面來展示它。

In [ ]:
import gradio as gr

def chat_session(user_message, history, system_prompt, top_p, temperature):
    # Construct messages for OpenAI API compatible format
    messages = []

    # 1. Handle system prompt
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    # 2. Add history (Gradio's history is a list of dicts with 'role' and 'content')
    for msg in history:
        # OpenRouter/OpenAI expects 'user' and 'assistant' roles
        role = 'user' if msg['role'] == 'user' else 'assistant'
        messages.append({"role": role, "content": str(msg['content'])})

    # Add the current user message
    messages.append({"role": "user", "content": user_message})

    try:
        completion = client.chat.completions.create(
            model=model_name,
            messages=messages,
            temperature=temperature,
            top_p=top_p,
            max_tokens=2048, # Optional: set a max_tokens
        )
        return completion.choices[0].message.content
    except Exception as e:
        print(f"Error sending message to OpenRouter: {e}")
        return f"發生錯誤: {e}"

# Define initial values for reset
INITIAL_SYSTEM_PROMPT = "使用繁體中文"
INITIAL_TOP_P = 0.9
INITIAL_TEMPERATURE = 0.7

with gr.Blocks() as demo:
    gr.Markdown("# My 聊天機器人")
    gr.Markdown("您可以設定系統提示(人格設定)、調整選字策略 (top_p) 和回應溫度 (temperature)。")

    with gr.Row():
        with gr.Column(scale=1):
            system_prompt_input = gr.Textbox(
                label="系統提示 (System Prompt)",
                placeholder="設定聊天機器人的人設，例如：你是一個樂觀的AI助理。",
                lines=3,
                value=INITIAL_SYSTEM_PROMPT
            )
            top_p_slider = gr.Slider(
                minimum=0.0, maximum=1.0, step=0.01,
                value=INITIAL_TOP_P, label="選字策略 (Top P)",
                info="調整詞彙選擇的多樣性。值越大，選擇越多樣"
            )
            temperature_slider = gr.Slider(
                minimum=0.0, maximum=1.0, step=0.01,
                value=INITIAL_TEMPERATURE, label="回應溫度 (Temperature)",
                info="調整回應的隨機性。值越低會产生更可预测的回应，值越高越发散热情。"
            )
            clear_all_btn = gr.Button("清除所有 (Clear All)")

        with gr.Column(scale=2):
            chatbot_component = gr.Chatbot(height=300)
            msg = gr.Textbox(placeholder="輸入您的訊息...", container=False, scale=7)
            send_btn = gr.Button("送出")

    # Define the response function for messages
    def respond(message, chat_history, system_prompt, top_p, temperature):
        # 呼叫 OpenRouter 獲取回應
        bot_message = chat_session(message, chat_history, system_prompt, top_p, temperature)

        # 新版字典格式更新 Gradio 聊天紀錄
        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": bot_message})
        return chat_history

    # Clear function
    def clear_all_states_func():
        return (
            [],                     # clear chatbot history
            INITIAL_SYSTEM_PROMPT,  # reset system_prompt
            INITIAL_TOP_P,          # reset top_p
            INITIAL_TEMPERATURE,    # reset temperature
            ""                      # clear message textbox
        )

    clear_all_btn.click(
        clear_all_states_func,
        outputs=[chatbot_component, system_prompt_input, top_p_slider, temperature_slider, msg]
    )

    # Setup interaction for message submission
    msg.submit(
        respond,
        [msg, chatbot_component, system_prompt_input, top_p_slider, temperature_slider],
        chatbot_component,
    ).then(
        lambda: gr.update(value=""),
        inputs=None,
        outputs=msg,
    )

    send_btn.click(
        respond,
        [msg, chatbot_component, system_prompt_input, top_p_slider, temperature_slider],
        chatbot_component,
    ).then(
        lambda: gr.update(value=""),
        inputs=None,
        outputs=msg,
    )

demo.launch(debug=True, share=True, theme="soft")

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ef4d43d4c37bcee8b6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
